# PalmSight V2 — Fine-tuning YOLO11n di Kaggle

Notebook ini memulai eksperimen **V2 dari `best.pt` PalmSight V1** yang sudah kamu download ke komputer. Ini adalah *fine-tuning / warm start*, bukan `resume=True`: optimizer dan hitungan epoch dimulai sebagai eksperimen baru.

## Sebelum menjalankan

1. Di Kaggle, aktifkan **GPU T4 x2**, **Internet ON**, dan **Persistence = Files only**.
2. Upload file `best.pt` lokal melalui panel kanan **Input → Upload**. Setelah upload, file akan dicari otomatis di `/kaggle/input` atau `/kaggle/working`.
3. Jika upload langsung tidak tersedia, upload `best.pt` ke Google Drive, set akses **Anyone with the link → Viewer**, lalu isi `BEST_PT_DRIVE_URL` pada Block 4.
4. Dataset tetap diambil dari Google Drive seperti notebook sebelumnya. Link dataset yang sebelumnya dipakai dipertahankan.
5. Jangan gunakan test set untuk memilih parameter training. Notebook ini membandingkan V1 dan V2 pada validation set, lalu menyediakan evaluasi test sebagai pemeriksaan akhir.

**Penting:** `Files only` membantu mempertahankan file di `/kaggle/working` antar-session, tetapi bukan backup permanen. Download hasil akhir dan simpan salinan di Google Drive/komputer.

## Rencana eksperimen

- V1: checkpoint `best.pt` yang sudah ada.
- V2: fine-tuning konservatif 30 epoch, AdamW, learning rate 0.0003.
- Checkpoint: `last.pt`, `best.pt`, dan snapshot setiap 5 epoch.
- Evaluasi: validation V1 vs V2, kemudian test V2.
- Export: `best.pt` V2 ke ONNX.

Parameter V2 adalah eksperimen awal, bukan jaminan Recall akan meningkat. Bandingkan hasil validation V1 dan V2 sebelum menyimpulkan V2 lebih baik.


## Block 1 — Persiapan environment

In [1]:
# Gunakan versi Ultralytics yang sama dengan eksperimen V1.
%pip install -q ultralytics==8.4.208 gdown


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import glob
import json
import shutil
import zipfile
import warnings
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak aktif. Aktifkan GPU di Kaggle Session Options.")

print("GPU 0:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Python: 3.13.16
Ultralytics: 8.4.208
PyTorch: 2.11.0+cu130
CUDA available: True
GPU count: 2
GPU 0: Tesla T4


## Block 2 — Path dan konfigurasi umum

In [3]:
# Folder kerja
WORK_DIR = Path("/kaggle/working")
DATASET_DIR = WORK_DIR / "dataset"
DATA_YAML = DATASET_DIR / "data.yaml"

# Dataset Google Drive dari notebook PalmSight sebelumnya.
# Jika link dataset berubah, ganti URL ini.
DATASET_DRIVE_URL = (
    "https://drive.google.com/file/d/"
    "1R2RD4N9jDkgoca4PXkt3xPgRFesaY7Vq/view?usp=sharing"
)
DATASET_ZIP = WORK_DIR / "dataset.zip"

# Model V1: isi path manual hanya jika auto-discovery tidak menemukan satu model yang jelas.
BEST_PT_PATH = ""

# Fallback jika upload langsung ke Kaggle tidak digunakan.
# Tempel Google Drive share link best.pt di sini, atau biarkan kosong.
BEST_PT_DRIVE_URL = ""

# Output V2
RUNS_DIR = WORK_DIR / "runs"
V2_RUN_NAME = "palm_sight_v2_finetune"
V2_BACKUP_DIR = WORK_DIR / "PalmSight_V2_Checkpoints"
V1_BACKUP_DIR = WORK_DIR / "PalmSight_V1_Backup"

for folder in [WORK_DIR, RUNS_DIR, V2_BACKUP_DIR, V1_BACKUP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("DATA_YAML:", DATA_YAML)


WORK_DIR: /kaggle/working
DATA_YAML: /kaggle/working/dataset/data.yaml


## Block 3 — Download dan ekstrak dataset

In [4]:
# Jika dataset sudah ada, jangan download/extract ulang.
if DATA_YAML.exists():
    print("Dataset sudah tersedia:", DATA_YAML)
else:
    import gdown

    if not DATASET_ZIP.exists():
        print("Mengunduh dataset dari Google Drive...")
        result = gdown.download(
            url=DATASET_DRIVE_URL,
            output=str(DATASET_ZIP),
            quiet=False,
            fuzzy=True,
        )
        if result is None or not DATASET_ZIP.exists():
            raise RuntimeError("Download dataset gagal. Periksa akses Google Drive.")

    if not zipfile.is_zipfile(DATASET_ZIP):
        raise ValueError("File dataset yang diunduh bukan ZIP valid.")

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_DIR)

    if not DATA_YAML.exists():
        yaml_candidates = list(DATASET_DIR.rglob("data.yaml"))
        if len(yaml_candidates) == 1:
            DATA_YAML = yaml_candidates[0]
        else:
            raise FileNotFoundError(
                "data.yaml tidak ditemukan atau ditemukan lebih dari satu: "
                f"{yaml_candidates}"
            )

print("Dataset YAML:", DATA_YAML)


Mengunduh dataset dari Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1R2RD4N9jDkgoca4PXkt3xPgRFesaY7Vq
From (redirected): https://drive.google.com/uc?id=1R2RD4N9jDkgoca4PXkt3xPgRFesaY7Vq&confirm=t&uuid=907f8f53-7ffe-44f5-a609-589970e2c5ff
To: /kaggle/working/dataset.zip
100%|██████████| 1.25G/1.25G [00:10<00:00, 115MB/s] 


Dataset YAML: /kaggle/working/dataset/data.yaml


## Block 4 — Temukan `best.pt` V1

Upload `best.pt` dari komputer lewat panel **Input → Upload**. Setelah upload, jalankan cell berikut. Bila memakai Google Drive, isi `BEST_PT_DRIVE_URL` pada Block 2 terlebih dahulu.

In [5]:
# Jika BEST_PT_PATH diisi, gunakan path tersebut.
# Jika kosong, cari file .pt yang namanya mengandung 'best' di Input/Working.
if BEST_PT_PATH.strip():
    BEST_PT = Path(BEST_PT_PATH).expanduser()
    if not BEST_PT.is_file():
        raise FileNotFoundError(f"BEST_PT_PATH tidak ditemukan: {BEST_PT}")
else:
    search_roots = [Path("/kaggle/input"), WORK_DIR]
    pt_candidates = []
    for root in search_roots:
        if root.exists():
            pt_candidates.extend(p for p in root.rglob("*.pt") if p.is_file())

    # Prioritaskan file yang namanya mengandung 'best'.
    best_candidates = [p for p in pt_candidates if "best" in p.name.lower()]

    if len(best_candidates) == 1:
        BEST_PT = best_candidates[0]
    elif len(best_candidates) > 1:
        print("Ditemukan beberapa kandidat best.pt:")
        for p in best_candidates:
            print(p)
        raise RuntimeError(
            "Isi BEST_PT_PATH di Block 2 dengan path checkpoint V1 yang benar."
        )
    elif len(pt_candidates) == 1:
        BEST_PT = pt_candidates[0]
    elif len(pt_candidates) > 1:
        print("Ditemukan beberapa file .pt:")
        for p in pt_candidates:
            print(p)
        raise RuntimeError(
            "Tidak ada nama file yang mengandung 'best'. Isi BEST_PT_PATH di Block 2."
        )
    else:
        BEST_PT = None

# Fallback download dari Google Drive bila tidak ada file lokal yang terdeteksi.
if BEST_PT is None:
    if not BEST_PT_DRIVE_URL.strip():
        raise FileNotFoundError(
            "best.pt belum ditemukan. Upload lewat Input → Upload, atau isi "
            "BEST_PT_DRIVE_URL di Block 2 dengan Google Drive share link."
        )

    import gdown
    BEST_PT = WORK_DIR / "PalmSight_V1_best.pt"
    gdown.download(
        url=BEST_PT_DRIVE_URL,
        output=str(BEST_PT),
        quiet=False,
        fuzzy=True,
    )
    if not BEST_PT.exists() or BEST_PT.stat().st_size == 0:
        raise RuntimeError("Download best.pt dari Google Drive gagal.")

print("V1 checkpoint:", BEST_PT)
print(f"Checkpoint size: {BEST_PT.stat().st_size / (1024**2):.2f} MB")


V1 checkpoint: /kaggle/input/models/hecatte/best-model-v1/pytorch/default/1/PalmSight_YOLO11n_best.pt
Checkpoint size: 5.26 MB


## Block 5 — Validasi dataset dan checkpoint V1

In [6]:
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"data.yaml tidak ditemukan: {DATA_YAML}")

# Validasi konfigurasi dataset dan path train/val/test.
dataset_info = check_det_dataset(str(DATA_YAML))
print("Train:", dataset_info.get("train"))
print("Validation:", dataset_info.get("val"))
print("Test:", dataset_info.get("test"))
print("Classes:", dataset_info.get("names"))
print("Number of classes:", dataset_info.get("nc"))

if int(dataset_info.get("nc", 0)) != 1:
    raise ValueError(f"Eksperimen ini mengharapkan 1 class, ditemukan {dataset_info.get('nc')}.")

if not dataset_info.get("test"):
    raise ValueError("data.yaml tidak mendefinisikan test split.")

# Load V1 checkpoint. Ini juga memvalidasi bahwa file .pt dapat dibaca Ultralytics.
v1_model = YOLO(str(BEST_PT))
print("V1 model names:", v1_model.names)
v1_model.info()

names = list(v1_model.names.values()) if isinstance(v1_model.names, dict) else list(v1_model.names)
if len(names) != 1 or names[0] != "palm_fruit":
    raise ValueError(f"Class model V1 tidak sesuai. Ditemukan: {names}")

# Simpan salinan V1 terpisah sebelum training V2.
V1_BACKUP = V1_BACKUP_DIR / "PalmSight_V1_best.pt"
if BEST_PT.resolve() != V1_BACKUP.resolve():
    shutil.copy2(BEST_PT, V1_BACKUP)
print("V1 backup:", V1_BACKUP)


Train: /kaggle/working/dataset/train/images
Validation: /kaggle/working/dataset/valid/images
Test: /kaggle/working/dataset/test/images
Classes: {0: 'palm_fruit'}
Number of classes: 1
V1 model names: {0: 'palm_fruit'}
YOLO11n summary: 182 layers, 2,604,883 parameters, 0 gradients, 6.6 GFLOPs
V1 backup: /kaggle/working/PalmSight_V1_Backup/PalmSight_V1_best.pt


## Block 6 — Baseline V1 pada validation set

Validation set dipakai untuk membandingkan V1 dan V2. Test set tidak dipakai untuk memilih parameter.

In [7]:
v1_val_metrics = v1_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    plots=True,
    project=str(WORK_DIR / "evaluation"),
    name="palm_sight_v1_validation",
    exist_ok=True,
)

V1_VAL = {
    "precision": float(v1_val_metrics.box.mp),
    "recall": float(v1_val_metrics.box.mr),
    "map50": float(v1_val_metrics.box.map50),
    "map50_95": float(v1_val_metrics.box.map),
}

print("V1 validation metrics:")
for key, value in V1_VAL.items():
    print(f"{key}: {value:.4f}")


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1738.8±1206.8 MB/s, size: 222.6 KB)
val: Scanning /kaggle/working/dataset/valid/labels... 913 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 913/913 1.1Kit/s 0.8s
val: New cache created: /kaggle/working/dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 6.2it/s 9.3s
                   all        913       4773      0.967      0.693      0.805      0.662
Speed: 0.9ms preprocess, 4.2ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /kaggle/working/evaluation/palm_sight_v1_validation
V1 validation metrics:
precision: 0.9670
recall: 0.6931
map50: 0.8053
map50_95: 0.6620


## Block 7 — Fine-tuning V2 dengan backup checkpoint

V2 dimulai dari `best.pt` V1 dengan optimizer baru. `resume=True` **tidak** digunakan karena `last.pt` V1 tidak tersedia. Callback menyalin `last.pt` dan `best.pt` ke folder backup pada setiap event penyimpanan checkpoint, serta membuat snapshot setiap 5 epoch.

In [8]:
def backup_v2_checkpoints(trainer):
    # Salin checkpoint setelah Ultralytics menyimpannya.
    try:
        epoch_number = int(trainer.epoch) + 1
        weights_dir = Path(trainer.save_dir) / "weights"
        V2_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

        for filename in ["last.pt", "best.pt"]:
            source = weights_dir / filename
            if source.exists():
                # Salinan terbaru yang terus diperbarui.
                shutil.copy2(source, V2_BACKUP_DIR / filename)

                # Snapshot tambahan setiap 5 epoch.
                if epoch_number % 5 == 0:
                    snapshot = V2_BACKUP_DIR / f"{source.stem}_epoch_{epoch_number:03d}.pt"
                    shutil.copy2(source, snapshot)

        print(f"Checkpoint backup updated at epoch {epoch_number}")
    except Exception as exc:
        # Jangan biarkan kegagalan backup callback menghentikan training.
        print(f"WARNING: checkpoint backup callback failed: {exc}")

# Muat kembali V1 agar V1 object untuk baseline tidak dipakai ulang.
v2_model = YOLO(str(BEST_PT))
v2_model.add_callback("on_model_save", backup_v2_checkpoints)

# Pastikan nama run belum dipakai agar path checkpoint tidak ambigu.
V2_RUN_DIR = RUNS_DIR / V2_RUN_NAME
if V2_RUN_DIR.exists():
    raise FileExistsError(
        f"Folder {V2_RUN_DIR} sudah ada. Ganti V2_RUN_NAME di Block 2 sebelum training."
    )

V2_RESULTS = v2_model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,

    # Fine-tuning konservatif: optimizer baru dengan learning rate lebih rendah.
    optimizer="AdamW",
    lr0=0.0003,
    lrf=0.01,
    patience=10,

    # Validasi dan checkpoint.
    val=True,
    plots=True,
    save=True,
    save_period=5,

    # Reproducibility.
    seed=42,
    deterministic=True,

    # Output terpisah; tidak menimpa V1.
    project=str(RUNS_DIR),
    name=V2_RUN_NAME,
    exist_ok=False,
)

print("V2 run directory:", V2_RUN_DIR)


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/hecatte/best-model-v1/pytorch/default/1/PalmSight_YOLO11n_best.pt, momentum=0.937, mosaic=1.0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      3.05G     0.8103     0.5232      1.104         24        640: 100% ━━━━━━━━━━━━ 596/596 7.5it/s 1:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.9it/s 4.9s
                   all        913       4773      0.952      0.702      0.796      0.647
Checkpoint backup updated at epoch 2

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      3.06G     0.8006       0.52      1.102         13        640: 100% ━━━━━━━━━━━━ 596/596 7.6it/s 1:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.8it/s 5.0s
                   all        913       4773      0.961      0.695      0.796      0.655
Checkpoint backup updated at epoch 3

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      3.06G     0.7864     0.5081      1.099          6        640: 100% ━━━━━━━━━━━━ 596

## Block 8 — Verifikasi dan backup checkpoint V2

In [9]:
V2_WEIGHTS_DIR = V2_RUN_DIR / "weights"
V2_BEST_PT = V2_WEIGHTS_DIR / "best.pt"
V2_LAST_PT = V2_WEIGHTS_DIR / "last.pt"

for path in [V2_BEST_PT, V2_LAST_PT]:
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {path}")
    print(f"{path.name}: {path.stat().st_size / (1024**2):.2f} MB")

# Backup final ke folder terpisah.
V2_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(V2_BEST_PT, V2_BACKUP_DIR / "best.pt")
shutil.copy2(V2_LAST_PT, V2_BACKUP_DIR / "last.pt")
print("Checkpoint final disalin ke:", V2_BACKUP_DIR)


best.pt: 5.25 MB
last.pt: 5.25 MB
Checkpoint final disalin ke: /kaggle/working/PalmSight_V2_Checkpoints


## Block 9 — Evaluasi V2 pada validation set dan bandingkan dengan V1

In [10]:
v2_best_model = YOLO(str(V2_BEST_PT))

v2_val_metrics = v2_best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    plots=True,
    project=str(WORK_DIR / "evaluation"),
    name="palm_sight_v2_validation",
    exist_ok=True,
)

V2_VAL = {
    "precision": float(v2_val_metrics.box.mp),
    "recall": float(v2_val_metrics.box.mr),
    "map50": float(v2_val_metrics.box.map50),
    "map50_95": float(v2_val_metrics.box.map),
}

print("\nV1 vs V2 — VALIDATION")
print(f"{'Metric':<12} {'V1':>10} {'V2':>10} {'Delta':>10}")
for key in ["precision", "recall", "map50", "map50_95"]:
    old = V1_VAL[key]
    new = V2_VAL[key]
    print(f"{key:<12} {old:>10.4f} {new:>10.4f} {new-old:>+10.4f}")

comparison = {"V1_validation": V1_VAL, "V2_validation": V2_VAL}
comparison_path = WORK_DIR / "PalmSight_V1_V2_validation_comparison.json"
comparison_path.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print("Comparison saved:", comparison_path)


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2502.3±2302.9 MB/s, size: 270.3 KB)
val: Scanning /kaggle/working/dataset/valid/labels.cache... 913 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 913/913 225.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 6.4it/s 9.0s
                   all        913       4773      0.964      0.696      0.811      0.662
Speed: 0.9ms preprocess, 4.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /kaggle/working/evaluation/palm_sight_v2_validation

V1 vs V2 — VALIDATION
Metric               V1         V2      Delta
precision        0.9670     0.9645    -0.0025
recall           0.6931     0.6962    +0.0031
map50            0.8053     0.8115    +0.0061
map50_95         0.6620     0.662

## Block 10 — Final test evaluation V2

Jalankan setelah membaca hasil validation V1 vs V2. Test split hanya untuk laporan akhir, bukan untuk memilih training parameters.

In [11]:
v2_test_metrics = v2_best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    plots=True,
    project=str(WORK_DIR / "evaluation"),
    name="palm_sight_v2_test",
    exist_ok=True,
)

V2_TEST = {
    "precision": float(v2_test_metrics.box.mp),
    "recall": float(v2_test_metrics.box.mr),
    "map50": float(v2_test_metrics.box.map50),
    "map50_95": float(v2_test_metrics.box.map),
}

print("\nPALMSIGHT V2 — TEST RESULTS")
for key, value in V2_TEST.items():
    print(f"{key}: {value:.4f}")

(WORK_DIR / "PalmSight_V2_test_metrics.json").write_text(
    json.dumps(V2_TEST, indent=2), encoding="utf-8"
)


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2171.4±2099.5 MB/s, size: 259.8 KB)
val: Scanning /kaggle/working/dataset/test/labels... 458 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 1.2Kit/s 0.4s
val: New cache created: /kaggle/working/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.8it/s 5.0s
                   all        458       2366       0.96      0.713      0.834      0.675
Speed: 0.9ms preprocess, 4.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/working/evaluation/palm_sight_v2_test

PALMSIGHT V2 — TEST RESULTS
precision: 0.9602
recall: 0.7133
map50: 0.8335
map50_95: 0.6752


134

## Block 11 — Export best.pt V2 ke ONNX

In [12]:
# Export dilakukan dari best.pt V2, bukan last.pt.
onnx_path = v2_best_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    device="cpu",
)

V2_ONNX = Path(onnx_path)
if not V2_ONNX.is_file():
    raise FileNotFoundError(f"ONNX tidak ditemukan: {V2_ONNX}")

print("ONNX exported:", V2_ONNX)
print(f"ONNX size: {V2_ONNX.stat().st_size / (1024**2):.2f} MB")


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs/palm_sight_v2_finetune/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 358ms
 Downloaded onnxruntime
Prepared 2 packages in 302ms
Installed 2 packages in 11ms
 + onnxruntime==1.31.1
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 1.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99

## Block 12 — Siapkan paket rilis dan backup

In [13]:
RELEASE_DIR = WORK_DIR / "PalmSight_V2_Release"
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

release_files = {
    "PalmSight_V2_best.pt": V2_BEST_PT,
    "PalmSight_V2_last.pt": V2_LAST_PT,
    "PalmSight_V2_best.onnx": V2_ONNX,
    "PalmSight_V1_best.pt": V1_BACKUP,
    "PalmSight_V1_V2_validation_comparison.json": comparison_path,
    "PalmSight_V2_test_metrics.json": WORK_DIR / "PalmSight_V2_test_metrics.json",
}

for output_name, source in release_files.items():
    source = Path(source)
    if source.exists():
        shutil.copy2(source, RELEASE_DIR / output_name)
        print("Added:", output_name)
    else:
        print("Skipped (not found):", source)

# Tambahkan log training dan grafik jika tersedia.
for filename in ["results.csv", "args.yaml", "results.png", "BoxPR_curve.png", "BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png", "confusion_matrix.png"]:
    candidates = [
        V2_RUN_DIR / filename,
        WORK_DIR / "evaluation" / "palm_sight_v2_validation" / filename,
        WORK_DIR / "evaluation" / "palm_sight_v2_test" / filename,
    ]
    for source in candidates:
        if source.exists():
            target_name = f"{source.parent.name}_{filename}"
            shutil.copy2(source, RELEASE_DIR / target_name)
            break

ARCHIVE_PATH = WORK_DIR / "PalmSight_V2_Complete_Backup.zip"
with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in RELEASE_DIR.iterdir():
        if file.is_file():
            archive.write(file, arcname=file.name)

print("\nRelease folder:", RELEASE_DIR)
print("Backup ZIP:", ARCHIVE_PATH)
print(f"ZIP size: {ARCHIVE_PATH.stat().st_size / (1024**2):.2f} MB")


Added: PalmSight_V2_best.pt
Added: PalmSight_V2_last.pt
Added: PalmSight_V2_best.onnx
Added: PalmSight_V1_best.pt
Added: PalmSight_V1_V2_validation_comparison.json
Added: PalmSight_V2_test_metrics.json

Release folder: /kaggle/working/PalmSight_V2_Release
Backup ZIP: /kaggle/working/PalmSight_V2_Complete_Backup.zip
ZIP size: 23.67 MB


## Block 13 — Download file hasil

In [14]:
from IPython.display import FileLink, display

for filename in [
    "PalmSight_V2_best.pt",
    "PalmSight_V2_last.pt",
    "PalmSight_V2_best.onnx",
]:
    filepath = RELEASE_DIR / filename
    if filepath.exists():
        print("DOWNLOAD:", filename)
        display(FileLink(str(filepath)))

print("DOWNLOAD COMPLETE BACKUP")
display(FileLink(str(ARCHIVE_PATH)))


DOWNLOAD: PalmSight_V2_best.pt


/kaggle/working/PalmSight_V2_Release/PalmSight_V2_best.pt

DOWNLOAD: PalmSight_V2_last.pt


/kaggle/working/PalmSight_V2_Release/PalmSight_V2_last.pt

DOWNLOAD: PalmSight_V2_best.onnx


/kaggle/working/PalmSight_V2_Release/PalmSight_V2_best.onnx

DOWNLOAD COMPLETE BACKUP


/kaggle/working/PalmSight_V2_Complete_Backup.zip

## Checklist setelah selesai

- [ ] Download `PalmSight_V2_best.pt` ke komputer.
- [ ] Download `PalmSight_V2_last.pt` ke komputer.
- [ ] Download `PalmSight_V2_best.onnx` ke komputer.
- [ ] Download `PalmSight_V2_Complete_Backup.zip`.
- [ ] Simpan salinan file di Google Drive.
- [ ] Simpan notebook menggunakan **Save Version** setelah run selesai.

**Catatan evaluasi:** Jika Recall V2 tidak membaik pada validation set, jangan otomatis lanjut menambah epoch. Periksa gambar yang miss, label, dan kondisi data. Fine-tuning pada dataset yang sama tidak menjamin peningkatan Recall.
